# Development of an Artificial Neural Network (ANN) for Binary Classification Using the `make_moons` Dataset

**Course:** Machine Learning Lab  |  **Tools:** Python, NumPy, Pandas, Matplotlib, Scikit-learn, TensorFlow/Keras

### Objectives
1. Understand binary classification with an ANN.
2. Generate a nonlinear dataset with `make_moons()`.
3. Preprocess and visualize the data.
4. Build and train an ANN with TensorFlow/Keras.
5. Evaluate it with standard metrics.
6. Visualize the learned decision boundary.
7. Test the model with new inputs and compare with Logistic Regression.

> **Google Colab users:** all packages are pre-installed. Just run the cells from top to bottom.

## Step 0: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)   # makes results reproducible
print("TensorFlow version:", tf.__version__)

## Step 1: Dataset Generation

`make_moons()` creates two interleaving half-circles that look like two crescent moons. Each point has **2 features** (x and y position) and a **label** (0 or 1) telling which moon it belongs to.
`noise=0.2` adds random scatter so the moons overlap slightly, like real data.

**Why is it good for nonlinear classification?** No single straight line can separate the two moons, so a linear model struggles while a neural network with nonlinear activations can learn the curved boundary.

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.2, random_state=SEED)

df = pd.DataFrame(X, columns=["feature_1", "feature_2"])
df["target"] = y
display(df.head())

print("Number of samples :", X.shape[0])
print("Number of features:", X.shape[1])
print(df["target"].value_counts())

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="tab:blue", edgecolor="k", label="Class 0")
plt.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:orange", edgecolor="k", label="Class 1")
plt.title("make_moons Dataset (1000 samples, noise=0.2)")
plt.xlabel("Feature 1"); plt.ylabel("Feature 2"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Step 2: Data Preprocessing

* **Train/test split (80:20):** the model learns from the training set and is judged on the unseen test set. `stratify=y` keeps the class ratio equal in both sets.
* **StandardScaler:** rescales each feature to mean 0 and standard deviation 1.

**Why scale for neural networks?** Neural networks learn with gradient descent. When features have different ranges, training becomes slow and unstable. Scaled inputs let the optimizer converge faster and more reliably.

**Important:** the scaler is fitted **only on training data**. Fitting on test data would leak information (data leakage).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled = scaler.transform(X_test)         # only transform on test

print("Training set:", X_train.shape, "| Testing set:", X_test.shape)
print("Scaled train mean:", X_train_scaled.mean(axis=0).round(3))
print("Scaled train std :", X_train_scaled.std(axis=0).round(3))

## Step 3: Build the ANN

| Layer | Details | Purpose |
|---|---|---|
| Input | 2 features | Receives (feature_1, feature_2) |
| Hidden 1 | 16 neurons, **ReLU** | Learns basic patterns |
| Hidden 2 | 8 neurons, **ReLU** | Combines them into more complex shapes |
| Output | 1 neuron, **sigmoid** | Outputs a probability between 0 and 1 |

* **ReLU** `f(x) = max(0, x)`: simple and fast. It adds the nonlinearity that allows curved boundaries.
* **Sigmoid** squashes the output into 0 to 1, read as P(class 1). Probability >= 0.5 means class 1.
* **Adam optimizer:** an adaptive version of gradient descent that updates the weights to reduce the error.
* **Binary cross-entropy loss:** measures how far predicted probabilities are from the true labels. It punishes confident wrong answers heavily.
* **Accuracy:** fraction of correct predictions, tracked during training.

In [ ]:
model = Sequential([
    Input(shape=(2,)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## Step 4: Train the Model

* **Epoch:** one full pass through the training data.
* **Batch size (32):** number of samples processed before the weights are updated.
* **Loss:** the error value the model tries to minimize.
* **Validation split (20% of the training data):** used to check performance on data not used for weight updates.
* **Early stopping:** stops training when validation loss has not improved for 10 epochs (`patience=10`) and restores the best weights, which helps prevent overfitting.

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=10,
                           restore_best_weights=True, verbose=1)

history = model.fit(X_train_scaled, y_train,
                    validation_split=0.2,
                    epochs=100, batch_size=32,
                    callbacks=[early_stop], verbose=1)

print("Epochs actually run:", len(history.history["loss"]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(history.history["accuracy"], label="Training accuracy")
ax[0].plot(history.history["val_accuracy"], label="Validation accuracy")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Accuracy")
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(history.history["loss"], label="Training loss")
ax[1].plot(history.history["val_loss"], label="Validation loss")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Binary cross-entropy")
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**How to read these curves:** falling loss and rising accuracy mean the model is learning. If training loss keeps falling while validation loss rises, the model is overfitting, which is what early stopping guards against.

## Step 5: Evaluate on the Unseen Test Set

* **Accuracy:** correct predictions / all predictions.
* **Precision:** of the points predicted as class 1, how many really are class 1.
* **Recall:** of all real class 1 points, how many were found.
* **F1-score:** the harmonic mean of precision and recall (a balance of both).
* **Confusion matrix:** a table of true vs. predicted classes (correct predictions on the diagonal, mistakes off the diagonal).

All values below are **calculated from the model's actual predictions**, none are hardcoded.

In [ ]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
y_prob = model.predict(X_test_scaled, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

ann_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
}
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
for k, v in ann_metrics.items():
    print(f"{k:<10}: {v:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)
print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))

ConfusionMatrixDisplay(cm, display_labels=["Class 0", "Class 1"]).plot(cmap="Blues")
plt.title("ANN Confusion Matrix (Test Data)")
plt.show()

**Interpretation:** compare the numbers above. Accuracy shows overall correctness. If precision is lower than recall, the model tends to label some class 0 points as class 1 (false positives); if recall is lower, it misses some class 1 points (false negatives). The confusion matrix shows exactly how many of each error occurred. Results vary slightly with the random seed and training setup.

## Step 6: Decision Boundary Visualization

We build a grid of points covering the whole plane, scale them with the **same fitted scaler**, and let the ANN predict a probability for every grid point. The colors show the predicted regions, and the black line is where the probability equals 0.5, i.e. the decision boundary.

**How does the ANN learn a curved boundary?** Each ReLU neuron creates a bent (piecewise-linear) function. Stacking many neurons across two layers combines these pieces into a smooth curve that wraps around the moons. A straight-line classifier such as Logistic Regression can only draw one straight boundary, so it cannot follow the crescent shapes.

In [ ]:
def plot_decision_boundary(predict_fn, title):
    """predict_fn takes SCALED inputs and returns the probability of class 1."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid)             # SAME scaling as training data
    probs = predict_fn(grid_scaled).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, probs, levels=np.linspace(0, 1, 21), cmap="RdBu_r", alpha=0.6)
    plt.colorbar(label="Predicted probability of Class 1")
    plt.contour(xx, yy, probs, levels=[0.5], colors="k", linewidths=2)
    plt.scatter(X[y == 0, 0], X[y == 0, 1], c="tab:blue", edgecolor="k", label="Class 0")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:red", edgecolor="k", label="Class 1")
    plt.title(title); plt.xlabel("Feature 1"); plt.ylabel("Feature 2"); plt.legend()
    plt.show()

plot_decision_boundary(lambda g: model.predict(g, verbose=0).ravel(),
                       "ANN Decision Boundary (black line = 0.5 threshold)")

## Step 7: Test New Inputs

The function below takes two raw (unscaled) numbers, scales them with the **already fitted** `scaler`, and returns the predicted class and probability. Class = 1 if probability >= 0.5, else 0. A probability near 0 or 1 means the model is confident, while a value near 0.5 means the point lies close to the boundary.

In [ ]:
def predict_new_point(feature_1, feature_2):
    point = np.array([[feature_1, feature_2]])
    point_scaled = scaler.transform(point)
    prob = float(model.predict(point_scaled, verbose=0)[0][0])
    return int(prob >= 0.5), prob

for f1, f2 in [(0.0, 0.5), (1.0, -0.5), (-1.0, 0.0), (2.0, 0.5)]:
    cls, prob = predict_new_point(f1, f2)
    print(f"Input ({f1:5.2f}, {f2:5.2f}) -> predicted class: {cls}, P(class 1) = {prob:.4f}")

## Step 8: Baseline Comparison with Logistic Regression

Logistic Regression separates classes with a **straight line**. We train it on the same scaled training data and test it on the same test set so the comparison is fair.

In [ ]:
log_reg = LogisticRegression(random_state=SEED)
log_reg.fit(X_train_scaled, y_train)
lr_pred = log_reg.predict(X_test_scaled)

lr_metrics = {
    "Accuracy": accuracy_score(y_test, lr_pred),
    "Precision": precision_score(y_test, lr_pred),
    "Recall": recall_score(y_test, lr_pred),
    "F1-score": f1_score(y_test, lr_pred),
}
comparison = pd.DataFrame({"ANN": ann_metrics, "Logistic Regression": lr_metrics}).round(4)
display(comparison)

comparison.plot(kind="bar", figsize=(8, 5), edgecolor="k")
plt.title("ANN vs Logistic Regression (Test Data)")
plt.ylabel("Score"); plt.ylim(0, 1.05); plt.xticks(rotation=0); plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
plot_decision_boundary(lambda g: log_reg.predict_proba(g)[:, 1],
                       "Logistic Regression Decision Boundary (a straight line)")

**Discussion:** the Logistic Regression boundary is a straight line, so it must cut through the curved moons and misclassifies points near the overlap. The ANN can bend its boundary to follow the shape of the data, so it usually scores higher on this dataset. Actual numbers depend on the dataset, random seed and training setup.

## Conclusion
* `make_moons` is a nonlinear binary classification problem.
* Scaling the inputs (fit on training data only) helps the network train properly.
* A small ANN (16-8-1) with ReLU/sigmoid, Adam and binary cross-entropy learns a curved boundary.
* Early stopping with `restore_best_weights=True` helps control overfitting.
* The ANN was compared against a linear baseline using metrics computed from real predictions.